Cell 1 - Imports

In [10]:
# ================================
# IMPORTS
# ================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from pyfaidx import Fasta

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, precision_recall_curve

from itertools import product
import tempfile

# ================================
# IMPORT MARKOV MODEL FUNCTIONS
# ================================

from MM import encode_sequences
from  MM import build_markov_model
from MM import score_sequences



Cell 2 - Paths and config

In [28]:
# ================================
# CONFIG (FIXED SPLIT)
# ================================

DATA_DIR = "/Users/dhruv/Documents/CFG/data"
BIN_DIR  = "/Users/dhruv/Documents/CFG/ref"

tf = "REST"
k = 4

# -------------------------------
# DEFINE CHROMOSOMES MANUALLY
# -------------------------------

# These have labeled data
train_chroms = [
    "chr1", "chr2", "chr4", "chr5", "chr6",
    "chr7", "chr8", "chr9", "chr11", "chr12",
    "chr13", "chr14", "chr15", "chr16",
    "chr19", "chr20", "chr21", "chr22"
]

# These are for prediction only
test_chroms = ["chr3", "chr10", "chr17"]

print("Train:", train_chroms)
print("Test:", test_chroms)

Train: ['chr1', 'chr2', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr19', 'chr20', 'chr21', 'chr22']
Test: ['chr3', 'chr10', 'chr17']


Cell 3 - Data Loading

In [29]:
# ================================
# LOAD DATA (ROBUST + INDEX TRACKING)
# ================================

def load_data(tsv_file, fasta_file, tf_name, has_labels=True):

    df = pd.read_csv(tsv_file, sep="\t")

    if has_labels:
        df = df[df[tf_name].isin(["B", "U"])]
        df["label"] = (df[tf_name] == "B").astype(int)
    else:
        df["label"] = -1

    df["ATAC_bin"] = (df["ATAC"] == "B").astype(int)

    # Fix FASTA header if missing
    with open(fasta_file, "r") as f:
        first_char = f.read(1)

    if first_char != ">":
        temp_fasta = tempfile.NamedTemporaryFile(delete=False, mode='w', suffix=".fa")

        chrom_name = os.path.basename(fasta_file).replace(".fa", "")
        temp_fasta.write(f">{chrom_name}\n")

        with open(fasta_file, "r") as original:
            for line in original:
                temp_fasta.write(line)

        temp_fasta.close()
        fasta_to_use = temp_fasta.name
    else:
        fasta_to_use = fasta_file

    genome = Fasta(fasta_to_use)
    chrom_name = list(genome.keys())[0]

    sequences, labels, atac, indices = [], [], [], []

    for i, row in df.iterrows():

        seq = genome[chrom_name][row["start"]:row["end"]].seq.upper()

        # 🚨 DROP sequences containing N
        if "N" in seq:
            continue

        sequences.append(seq)
        labels.append(row["label"])
        atac.append(row["ATAC_bin"])
        indices.append(i)

    return np.array(sequences), np.array(labels), np.array(atac), np.array(indices)

Cell 4 - K-mer Features

In [ ]:
# # ================================
# # K-MER FEATURES
# # ================================

# def generate_kmers(k):
#     return [''.join(p) for p in product('ACGT', repeat=k)]


# def kmer_features(sequences, k):

#     kmers = generate_kmers(k)
#     kmer_index = {kmer: i for i, kmer in enumerate(kmers)}

#     X = np.zeros((len(sequences), len(kmers)))

#     for i, seq in enumerate(sequences):
#         for j in range(len(seq) - k + 1):
#             kmer = seq[j:j+k]
#             if kmer in kmer_index:
#                 X[i, kmer_index[kmer]] += 1

#     X = X / X.sum(axis=1, keepdims=True)

#     return X


# def build_features(sequences, atac, k):
#     X_kmer = kmer_features(sequences, k)
#     atac = atac.reshape(-1, 1)
#     return np.hstack([X_kmer, atac])

K-mer/2 Features

In [30]:
# ================================
# K-MER FEATURES (STRAND-INVARIANT)
# ================================

def reverse_complement(seq):
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(complement[b] for b in reversed(seq))


def canonical_kmer(kmer):
    """
    Return lexicographically smaller of:
    kmer and its reverse complement
    """
    rc = reverse_complement(kmer)
    return min(kmer, rc)


def generate_canonical_kmers(k):
    """
    Generate unique k-mers considering reverse complement equivalence
    """
    kmers = set()

    for p in product('ACGT', repeat=k):
        kmer = ''.join(p)
        kmers.add(canonical_kmer(kmer))

    return sorted(list(kmers))


def kmer_features(sequences, k):

    # Generate canonical kmers
    kmers = generate_canonical_kmers(k)

    # Mapping
    kmer_index = {kmer: i for i, kmer in enumerate(kmers)}

    X = np.zeros((len(sequences), len(kmers)))

    for i, seq in enumerate(sequences):

        for j in range(len(seq) - k + 1):

            kmer = seq[j:j+k]

            # Skip k-mers containing N
            if "N" in kmer:
                continue
            
            # Convert to canonical form
            canon = canonical_kmer(kmer)

            if canon in kmer_index:
                X[i, kmer_index[canon]] += 1

    # Normalize
    row_sums = X.sum(axis=1, keepdims=True)

    # Avoid division by zero
    row_sums[row_sums == 0] = 1

    X = X / row_sums  
      
    return X


def build_features(sequences, atac, k):
    X_kmer = kmer_features(sequences, k)
    atac = atac.reshape(-1, 1)
    return np.hstack([X_kmer, atac])

Cell 5 - Training + Cross Validation

In [31]:
def train_and_validate(sequences, atac, y, k, m=6):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    roc_aucs, pr_aucs = [], []

    # Encode sequences ONCE
    encoded_all = encode_sequences(sequences)

    plt.figure(figsize=(12,5))

    for i, (train_idx, val_idx) in enumerate(skf.split(sequences, y)):

        print(f"\n=== Fold {i+1} ===")

        # ------------------------
        # SPLIT
        # ------------------------
        seq_train, seq_val = sequences[train_idx], sequences[val_idx]
        atac_train, atac_val = atac[train_idx], atac[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        enc_train = encoded_all[train_idx]
        enc_val   = encoded_all[val_idx]

        # ------------------------
        # K-MER FEATURES
        # ------------------------
        X_train_kmer = build_features(seq_train, atac_train, k)
        X_val_kmer   = build_features(seq_val, atac_val, k)

        # ------------------------
        # MARKOV MODEL (TRAIN ONLY)
        # ------------------------
        probs_B = build_markov_model(enc_train[y_train == 1], m)
        probs_U = build_markov_model(enc_train[y_train == 0], m)

        # ------------------------
        # MM SCORES
        # ------------------------
        mm_train = score_sequences(enc_train, probs_B, probs_U, m)
        mm_val   = score_sequences(enc_val, probs_B, probs_U, m)

        mm_train = mm_train.reshape(-1, 1)
        mm_val   = mm_val.reshape(-1, 1)

        # ------------------------
        # COMBINE FEATURES
        # ------------------------
        X_train = np.hstack([X_train_kmer, mm_train])
        X_val   = np.hstack([X_val_kmer, mm_val])

        # ------------------------
        # SCALE (IMPORTANT)
        # ------------------------
        from sklearn.preprocessing import StandardScaler

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)

        # ------------------------
        # TRAIN MODEL
        # ------------------------
        model = LogisticRegression(max_iter=1000, class_weight='balanced')
        model.fit(X_train, y_train)

        probs = model.predict_proba(X_val)[:,1]

        # ---- ROC ----
        fpr, tpr, _ = roc_curve(y_val, probs)
        roc_auc = auc(fpr, tpr)
        roc_aucs.append(roc_auc)

        plt.subplot(1,2,1)
        plt.plot(fpr, tpr, alpha=0.4, label=f"Fold {i+1} (AUC={roc_auc:.2f})")

        # ---- PR ----
        precision, recall, _ = precision_recall_curve(y_val, probs)
        pr_auc = auc(recall, precision)
        pr_aucs.append(pr_auc)

        plt.subplot(1,2,2)
        plt.plot(recall, precision, alpha=0.4, label=f"Fold {i+1} (AUC={pr_auc:.2f})")

    # ---- ROC ----
    plt.subplot(1,2,1)
    plt.plot([0,1], [0,1], linestyle='--')
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"ROC (Mean AUC = {np.mean(roc_aucs):.3f})")
    plt.legend()

    # ---- PR ----
    plt.subplot(1,2,2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR (Mean AUC = {np.mean(pr_aucs):.3f})")
    plt.legend()

    plt.tight_layout()
    plt.show()

    return model


# def train_full_model(X, y):
#     model = LogisticRegression(max_iter=1000)
#     model.fit(X, y)
#     return model


def predict(model, sequences, atac, k):
    X = build_features(sequences, atac, k)
    return model.predict_proba(X)[:,1]

Cell 6 - Load Training Data

In [32]:
# ================================
# LOAD TRAINING DATA
# ================================

sequences_all, labels_all, atac_all = [], [], []

for chrom in train_chroms:

    fasta_file = f"{DATA_DIR}/{chrom}.fa"
    tsv_file   = f"{BIN_DIR}/{chrom}_200bp_bins.tsv"

    seqs, labs, atac, _ = load_data(tsv_file, fasta_file, tf, has_labels=True)

    sequences_all.extend(seqs)
    labels_all.extend(labs)
    atac_all.extend(atac)

sequences_all = np.array(sequences_all)
labels_all    = np.array(labels_all)
atac_all      = np.array(atac_all)

print("Training samples:", len(sequences_all))
print("Positive fraction:", np.mean(labels_all))

Training samples: 3479538
Positive fraction: 0.011241147531655065


Cell 7 - Features

Feature matrix shape: (3479552, 137)


Cell 8 - Cross Validation

In [ ]:
model = train_and_validate(sequences_all, atac_all, labels_all, k, m=6)


=== Fold 1 ===


Cell 9 - Final Model Training

In [20]:
def train_full_model(sequences, atac, y, k, m=6):

    # k-mer features
    X_kmer = build_features(sequences, atac, k)

    # encode
    encoded = encode_sequences(sequences)

    # build MM
    probs_B = build_markov_model(encoded[y == 1], m)
    probs_U = build_markov_model(encoded[y == 0], m)

    # MM scores
    mm_scores = score_sequences(encoded, probs_B, probs_U, m).reshape(-1,1)

    # combine
    X = np.hstack([X_kmer, mm_scores])

    # scale
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X, y)

    return model, scaler, probs_B, probs_U

model, scaler, probs_B, probs_U = train_full_model(
    sequences_all, atac_all, labels_all, k, m=6
)

In [ ]:
# ================================
# PREDICTION FUNCTION (CONSISTENT WITH TRAINING)
# ================================

def predict(model, scaler, sequences, atac, k, probs_B, probs_U, m=6):

    # ------------------------
    # K-MER FEATURES (+ ATAC already included)
    # ------------------------
    X_kmer = build_features(sequences, atac, k)

    # ------------------------
    # ENCODE SEQUENCES FOR MM
    # ------------------------
    encoded = encode_sequences(sequences)

    # ------------------------
    # COMPUTE MARKOV SCORES
    # ------------------------
    mm_scores = score_sequences(encoded, probs_B, probs_U, m)

    # optional but safe
    mm_scores = np.clip(mm_scores, -50, 50)

    mm_scores = mm_scores.reshape(-1, 1)

    # ------------------------
    # COMBINE FEATURES (NO interaction!)
    # ------------------------
    X = np.hstack([X_kmer, mm_scores])

    # ------------------------
    # APPLY SAME SCALING
    # ------------------------
    X = scaler.transform(X)

    # ------------------------
    # PREDICT
    # ------------------------
    return model.predict_proba(X)[:, 1]

Cell 10 - Testing Unknown Data

In [27]:
# ================================
# TESTING + WRITE BACK TO TSV
# ================================

for chrom in test_chroms:

    print(f"\nProcessing {chrom}...")

    fasta_file = f"{DATA_DIR}/{chrom}.fa"
    tsv_file   = f"{BIN_DIR}/{chrom}_200bp_bins_unknown.tsv"

    # Load full dataframe
    df = pd.read_csv(tsv_file, sep="\t")

    # Load filtered sequences + indices
    seqs, _, atac, indices = load_data(tsv_file, fasta_file, tf, has_labels=False)

    # Predict
    probs = predict(
        model,
        scaler,
        seqs,
        atac,
        k,
        probs_B,
        probs_U,
        m=6
    )

    # -----------------------------
    # WRITE BACK INTO CTCF COLUMN
    # -----------------------------

    # Create column if it doesn't exist
    if tf not in df.columns:
        df[tf] = np.nan

    # Assign predictions to correct rows
    df.loc[indices, tf] = probs

    # Optional: keep NaN (recommended) OR fill
    # df[tf] = df[tf].fillna(0)

    # Save back to same file
    df.to_csv(tsv_file, sep="\t", index=False)

    print(f"{chrom}: updated {len(probs)} rows in {tsv_file}")


Processing chr3...
chr3: updated 307938 rows in /Users/dhruv/Documents/CFG/ref/chr3_200bp_bins_unknown.tsv

Processing chr10...
chr10: updated 212977 rows in /Users/dhruv/Documents/CFG/ref/chr10_200bp_bins_unknown.tsv

Processing chr17...
chr17: updated 122524 rows in /Users/dhruv/Documents/CFG/ref/chr17_200bp_bins_unknown.tsv
